In [9]:
import pandas as pd
from pathlib import Path
from enums import Frequency
import pickle

In [10]:
save_to_file = 'categorized_timeseries_dict.pkl'

In [11]:
def get_frequency(df, filename):
    file_number = int(filename.split('_')[1])
    if file_number in [25399, 25486, 25459, 25449, 25261, 25386, 25347, 25312]:
        return Frequency.WEEKLY
    elif file_number >= 29494:
        return Frequency.HOURLY
    else:
        index_space_split = str(df.index[0]).split(' ')
        if len(index_space_split) > 1:
            if index_space_split[1] in ['Q1', 'Q2', 'Q3', 'Q4']:
                return Frequency.QUARTERLY
            elif index_space_split[1] in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']:
                return Frequency.MONTHLY
            else:
                raise Exception('Unexpected Format')
        else:
            if any(1800 <= x <= 2026 for x in df.index.to_list()):
                return Frequency.ANNUALLY
            else:
                return Frequency.DAILY

def str_to_float(s):
    if type(s) is str:
        return float(s.replace(",", "."))
    else:
        return s

In [12]:
folder = Path('data_training')
ts_all = {}

# save classified frequency counts to check consistency later 
frequency_count = {
    Frequency.HOURLY: 0,
    Frequency.DAILY: 0, 
    Frequency.WEEKLY: 0,
    Frequency.MONTHLY: 0,
    Frequency.QUARTERLY: 0,
    Frequency.ANNUALLY: 0}


for file in folder.glob("*.csv"):
    df = pd.read_csv(
        file,
        index_col="index",
        sep=';'
    )
    ts = {}
    filename = file.stem
    df['value'] = df['value'].apply(str_to_float)
    ts['original'] = df
    freq = get_frequency(df, filename)
    ts['frequency'] = freq
    frequency_count[freq] = frequency_count.get(freq, 0) + 1
    ts_all[filename] = ts

##################################################################
### CHECK FREQUENCY COUNTS BY COMPARING TO PROJECT INFORMATION ###
##################################################################

expected = {
    Frequency.HOURLY: 15,
    Frequency.DAILY: 150,
    Frequency.WEEKLY: 8,
    Frequency.MONTHLY: 801,
    Frequency.QUARTERLY: 21,
    Frequency.ANNUALLY: 5,
}

print(f"{'Frequency':<12} {'Expected':>12} {'Identified':>12} {'Match':>8}")
print("-" * 50)
check = True
for freq, exp in expected.items():
    act = frequency_count.get(freq, 0)
    if act != exp:
        check = False
    status = "✓" if act == exp else "✗"
    print(f"{freq.name:<12} {exp:>12} {act:>12} {status:>8}")
print("-" * 50)
if check:
    print('✓ CHECK PASSED')
    with open(save_to_file, 'wb') as f:
        pickle.dump(ts_all, f)
        print(f'Saved categorized timeseries data to {save_to_file}')

else:
    print('✗ CHECK FAILED')


Frequency        Expected   Identified    Match
--------------------------------------------------
HOURLY                 15           15        ✓
DAILY                 150          150        ✓
WEEKLY                  8            8        ✓
MONTHLY               801          801        ✓
QUARTERLY              21           21        ✓
ANNUALLY                5            5        ✓
--------------------------------------------------
✓ CHECK PASSED
